# 06 · Quantization / Distill / Prune —— 行李箱压缩三件套

**家族位置**：08 生产级优化第 6 站。01-05 让训练推理省算省存，本章把模型本身压小：量化（低比特存）、剪枝（零掉不重要的）、蒸馏（小模型学大模型）。

**学习目标**：PTQ 对称 INT8 的 scale/反量化；全局幅度剪枝 mask；蒸馏 T²·KL+CE；体积/精度三口径对照。

## 1. 原理：叠衣服、扔衣服、学着装

### 通俗理解

**一句话**：量化像把衣服抽真空（fp32→INT8，体积÷4，样子基本不变）；剪枝像扔不穿的（小权重直接置零）；蒸馏像小箱学大箱装（学生只看老师软标签就学会）。

### 结构账

```
老师： TinyMNIST ~100k 参，MNIST 8000 条 5ep（fp32 基准）
量化： 逐张量对称 INT8 PTQ：scale=max|w|/127，q=round(w/scale)，体积≈1B/参
剪枝： 全局幅度：最小 |w| 的 s% 置零（只剪 Linear/Conv）；扫 s=0/0.3/0.5/0.7/0.9
蒸馏： 学生通道减半 ~27k 参；L=α·T²·KL+T=4, α=0.7；学生单训 vs 蒸馏对照
```

In [ ]:
import sys, copy
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import load_mnist_local
from common.models import TinyMNIST, TinyMNISTSmall, model_bytes_fp32, quantize_int8_per_tensor, dequantize_state, quantized_bytes, global_magnitude_mask, apply_mask
from common.engine import fit_mnist, fit_distill, mnist_accuracy
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xtr,ytr,Xva,yva,Xte,yte=load_mnist_local(8000,1000,seed=0)
tr=DataLoader(TensorDataset(Xtr,ytr),batch_size=128,shuffle=True)
te=DataLoader(TensorDataset(Xte,yte),batch_size=512)
print(f'train {tuple(Xtr.shape)} / test {tuple(Xte.shape)} | MNIST 标准化域')

## 2. 基准 + INT8 PTQ：体积÷4，精度几乎不掉

In [ ]:
torch.manual_seed(0)
teacher=TinyMNIST()
fit_mnist(teacher,tr,epochs=5,lr=1e-3)
acc_fp32=mnist_accuracy(teacher,te)
fp32b=model_bytes_fp32(teacher)
print(f'teacher params={count_params(teacher)} fp32={fp32b/1e6:.2f}MB acc={acc_fp32:.4f}',flush=True)
qstate=quantize_int8_per_tensor(teacher)
int8b=quantized_bytes(qstate)
qi=copy.deepcopy(teacher)
qi.load_state_dict(dequantize_state(qstate))
acc_int8=mnist_accuracy(qi,te)
print(f'INT8 되는={int8b/1e6:.2f}MB (×{fp32b/int8b:.2f}) acc={acc_int8:.4f} (Δ={acc_int8-acc_fp32:+.4f})',flush=True)
fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].bar(['fp32','INT8'],[fp32b/1e6,int8b/1e6],color=['#4C72B0','#55A868'])
for i,v in enumerate([fp32b/1e6,int8b/1e6]): ax[0].text(i,v+0.01,f'{v:.2f}MB',ha='center')
ax[0].set_title('体积：fp32→INT8'); ax[0].set_ylabel('MB')
ax[1].bar(['fp32','INT8'],[acc_fp32,acc_int8],color=['#4C72B0','#55A868'])
for i,v in enumerate([acc_fp32,acc_int8]): ax[1].text(i,v+0.002,f'{v:.4f}',ha='center')
ax[1].set_ylim(min(acc_fp32,acc_int8)-0.01,1.0); ax[1].set_title('精度：PTQ 几乎不掉')
plt.tight_layout(); plt.savefig(FIGS/'fig1_quant.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 剪枝：稀疏率-精度曲线

In [ ]:
sps=[0.0,0.3,0.5,0.7,0.9]; accs=[]
for s in sps:
    m=copy.deepcopy(teacher)
    if s>0:
        masks,_=global_magnitude_mask(m,s); real=apply_mask(m,masks)
    else: real=0.0
    a=mnist_accuracy(m,te); accs.append(a)
    print(f'sparsity目标={s:.1f} 实际={real:.3f} acc={a:.4f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.plot(sps,accs,marker='o',color='#DD8452')
ax.set_xlabel('sparsity'); ax.set_ylabel('test acc'); ax.set_ylim(min(accs)-0.02,1.0)
ax.set_title('全局幅度剪枝：s=0.5 前精度 почти不掉')
plt.tight_layout(); plt.savefig(FIGS/'fig2_prune.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 蒸馏：27k 学生 vs 100k 老师 + 三件套总览

In [ ]:
torch.manual_seed(0)
stu_scratch=TinyMNISTSmall()
fit_mnist(stu_scratch,tr,epochs=5,lr=1e-3)
acc_scratch=mnist_accuracy(stu_scratch,te)
torch.manual_seed(0)
stu_dist=TinyMNISTSmall()
fit_distill(stu_dist,teacher,tr,epochs=5,lr=1e-3,T=4.0,alpha=0.7)
acc_dist=mnist_accuracy(stu_dist,te)
print(f'student params={count_params(stu_dist)} ({count_params(stu_dist)/count_params(teacher)*100:.1f}% of teacher)',flush=True)
print(f'scratch acc={acc_scratch:.4f} | distill acc={acc_dist:.4f} | teacher acc={acc_fp32:.4f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(['teacher\n100k','student\n单训27k','student\n蒸馏27k'],[acc_fp32,acc_scratch,acc_dist],color=['#4C72B0','#DD8452','#55A868'])
for i,v in enumerate([acc_fp32,acc_scratch,acc_dist]): ax.text(i,v+0.002,f'{v:.4f}',ha='center')
ax.set_ylim(min(acc_scratch,acc_dist)-0.02,1.0); ax.set_title('蒸馏：小模型吃软标签追上老师')
plt.tight_layout(); plt.savefig(FIGS/'fig3_distill.png',dpi=150,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(7,3.4))
ax.axis('off')
ax.text(0.02,0.8,f'量化：{fp32b/1e6:.2f}MB→{int8b/1e6:.2f}MB (×{fp32b/int8b:.1f})，acc Δ={acc_int8-acc_fp32:+.4f}',fontsize=10)
ax.text(0.02,0.55,f'剪枝：s=0.5 acc={accs[2]:.4f}（dense {acc_fp32:.4f}），s=0.9 acc={accs[4]:.4f} 崩',fontsize=10)
ax.text(0.02,0.3,f'蒸馏：27k 学生 {acc_dist:.4f} vs 老师 {acc_fp32:.4f}（gap={acc_fp32-acc_dist:.4f}）',fontsize=10)
ax.text(0.02,0.05,'三件套可叠加：剪枝→量化→蒸馏（生产 QLoRA 即量化+LoRA 叠加，见 08-04）',fontsize=10,color='#1a6b3c')
ax.set_title('08-06 总览：体积/精度对照表')
plt.tight_layout(); plt.savefig(FIGS/'fig4_compare.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',round(acc_fp32,4),round(acc_int8,4),[round(a,4) for a in accs],round(acc_scratch,4),round(acc_dist,4))